### Simulación #2: Gestión de una bodega de datos

**Paso 1.** Configurar las rutas de Python para poder hacer uso de la implementación del *Piloto de Framework*

In [1]:
import sys
import os.path
from pathlib import Path
sys.path.append(str(Path(os.path.abspath('')).parent))
import logging
logger = logging.getLogger("root")
logger.setLevel(logging.ERROR)
#logger.setLevel(logging.WARNING)

**Paso 2.** Importar las librerías que conforman la implementación del *Piloto de Framework*

In [2]:
import re
import sqlalchemy
import pandas as pd
from datetime import datetime

from framework.integraciones.piloto_framework import piloto_framework
from framework.modelos.repositorio_datos import EstadoEjecucion, TipoCarga
import dotenv
dotenv.load_dotenv()

True

**Paso 3.** Configurar las variables básicas necesarias para el funcionamiento del *Piloto de Framework*
* Definir las variables asociadas a los identificadores de los procesos que conforman la simunación. Equivale a *PROCESO.ID_PROCESO* del *Repositorio de datos*
* Definir la variable **parametros** con el listado de identificadores de los parámetros a usar. Equivale a *PARAMETRO.ID_PARAMETRO* del *Repositorio de datos*

In [3]:
id_proceso_principal = "e3153df7-92ff-448a-ac5b-61031adfa4f3"
id_proceso_dim_entidad_pub = "317d33d8-025f-4eb1-baea-097873cc306d"
id_proceso_dim_entidad_sec = "57a54413-72e6-4a9e-bcb7-61044f447401"
id_proceso_dim_departamento = "83669434-13d7-40e3-92e3-fd67e24718e1"
id_proceso_fact_procjud_det = "782045a8-59f1-4411-9a7c-145eeefb6d2d"
id_proceso_fact_procjud_res = "e72b69f0-d128-440a-ad97-03616d9b85c6"

parametros = [
    "tabla_destino_dim_entidad_pub",
    "tabla_destino_dim_entidad_sec",
    "tabla_destino_dim_departamento",
    "tabla_destino_p1",
    "tabla_destino_hechos_procjud_det",
    "tabla_destino_hechos_procjud_res",
]

**Paso 4.** Ejecutar los métodos necesarios para realizar la configuración inicial del proceso principal y los parámetros

In [4]:
proc_principal = piloto_framework()
proc_principal.configurar_proceso(id_proceso_principal)
proc_principal.ver_datos_proceso()
proc_principal.configurar_parametros(parametros)
proc_principal.ver_parametros()

╭───────────────────────────────────────────── Detalles del proceso ──────────────────────────────────────────────╮
│ {                                                                                                               │
│   "nombre_proceso": "Procesar bodega de datos",                                                                 │
│   "tipo_carga_proceso": "Carga Inicial",                                                                        │
│   "fecha_fin_extraccion": null,                                                                                 │
│   "observaciones": null,                                                                                        │
│   "estado_registro": "Activo",                                                                                  │
│   "fecha_creacion": "2026-01-31 12:14:24",                                                                      │
│   "descripcion_proceso": "Proceso que gestiona una bodega de datos a partir de la información de los procesos j │
│   "id_proceso": "e3153df7-92ff-448a-ac5b-61031adfa4f3",                                                         │
│   "id_proceso_padre": null,                                                                                     │
│   "fecha_inicio_extraccion": null,                                                                              │
│   "email_responsable_proceso": "sirghoro@gmail.com",                                                            │
│   "estado_ultima_ejecucion": "Correcto",                                                                        │
│   "usuario_responsable": "postgres",                                                                            │
│   "fecha_ultima_modificacion": null                                                                             │
│ }                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────── Parámetros ────────────────────────────────────╮
│ {                                                                                 │
│   "tabla_destino_dim_entidad_pub": "simulacion.dim_entidad_publica",              │
│   "tabla_destino_dim_entidad_sec": "simulacion.dim_entidad_seccional",            │
│   "tabla_destino_dim_departamento": "simulacion.dim_departamento",                │
│   "tabla_destino_p1": "simulacion.procesos_judiciales_colombia",                  │
│   "tabla_destino_hechos_procjud_det": "simulacion.fact_proceso_judicial_detalle", │
│   "tabla_destino_hechos_procjud_res": "simulacion.fact_proceso_judicial_resumen"  │
│ }                                                                                 │
╰───────────────────────────────────────────────────────────────────────────────────╯

**Paso 5.** Simular los procesos ETL.
Este proceso simula la creación de 3 dimensiones de datos (ver [Dimensions of the data model](https://www.ibm.com/docs/en/informix-servers/12.10.0?topic=modeling-dimensions-data-model)) y 2 tablas de hechos (ver [The fact table](https://www.ibm.com/docs/en/informix-servers/12.10.0?topic=modeling-fact-table)) a partir de la información de los proceso judiciales de Colombia.
Este proceso aprovecha las funcionalidades del *Piloto de Framework* así:
* Hace uso de la gestión de **parámetros** para personalizar el nombre de la tabla destino de cada dimensión.
* Hace uso de la gestión de **procesos** para controlar las características básicas de cada proceso. Para esta simulación se gestionan 4 procesos: *3 procesos* asociados a las 3 dimensiones de datos a simular (*Entidad Pública, Entidad Seccional y Departamento*) y *1 proceso* que agrupa los procesos antes mencionados.

Al finalizar la ejecución del proceso ETL, el **Piloto de Framework** permite registrar los datos de particulares de la ejecución de cada proceso, tales como el *momento de inicio y fin* del proceso, el *estado de la ejecución* o la *cantidad de registros procesados*

**Paso 5.1.** Simular el proceso ETL para la dimensión *Entidad Pública*.

In [5]:
inicio_ejecucion_principal = datetime.now()
inicio_ejecucion = datetime.now()
mensaje = None
log_dim_entpub = None

proc_dim_entpub = piloto_framework()
proc_dim_entpub.configurar_proceso(id_proceso_dim_entidad_pub)
proc_dim_entpub.ver_datos_proceso()

if proc_dim_entpub.proceso._error_precedencias:
    log_dim_entpub = proc_dim_entpub.registro_log(
        inicio_ejecucion,
        EstadoEjecucion.ERROR,
        "Error de ejecución por precedencias: verifique el estado de la última ejecución de los procesos precedentes.",
        0,
    )
if proc_dim_entpub.proceso._ejecutar:
    try:
        par_tabla_fuente = proc_principal.parametros.get("tabla_destino_p1", "")
        par_tabla_destino = proc_principal.parametros.get(
            "tabla_destino_dim_entidad_pub", ""
        )
        consulta = f"select * from {par_tabla_fuente}"
        datos_fuente = pd.read_sql(consulta, con=proc_dim_entpub.bd_motor)
        entidades_publicas = datos_fuente[
            ["id_de_la_entidad", "entidad"]
        ].drop_duplicates()
        # entidades_publicas["sk_entidad_publica"] = [uuid.uuid4() for _ in range(len(entidades_publicas.index))]
        entidades_publicas = entidades_publicas.assign(
            sk_entidad_publica=range(1, len(entidades_publicas) + 1)
        )
        entidades_publicas["fecha_creacion"] = datetime.now()
        entidades_publicas["fecha_ultima_modificacion"] = None
        entidades_publicas.rename(
            columns={
                "id_de_la_entidad": "id_entidad_publica",
                "entidad": "nombre_entidad_publica",
            },
            inplace=True,
        )
        columnas = [
            "sk_entidad_publica",
            "id_entidad_publica",
            "nombre_entidad_publica",
            "fecha_creacion",
            "fecha_ultima_modificacion",
        ]
        entidades_publicas = entidades_publicas[columnas]
        grupo_1 = re.search(r"(?<=[.])\w+", par_tabla_destino)
        grupo_2 = re.search(r"(?<![.])\w+", par_tabla_destino)
        tabla: str | None = None
        esquema: str | None = None
        if grupo_1:
            tabla = grupo_1.group(0)
            esquema = grupo_2.group(0)
        elif grupo_2:
            tabla = grupo_2.group(0)
        else:
            tabla = ""
        tipo_carga = (
            "append"
            if proc_dim_entpub.proceso.tipo_carga_proceso == TipoCarga.INC
            else "replace"
        )
        entidades_publicas.to_sql(
            name=tabla,
            schema=esquema,
            con=proc_dim_entpub.bd_motor,
            if_exists=tipo_carga,
            index=False,
        )
        estado_ejecucion = EstadoEjecucion.CORRECTO
        registros_procesados = len(entidades_publicas)
    except Exception as error:
        estado_ejecucion = EstadoEjecucion.ERROR
        mensaje = f"Error en la ejecución: {error}"
        registros_procesados = 0
    finally:
        log_dim_entpub = proc_dim_entpub.registro_log(
            inicio_ejecucion, estado_ejecucion, mensaje, registros_procesados
        )

╭────────────────────────────────── Detalles del proceso ───────────────────────────────────╮
│ {                                                                                         │
│   "nombre_proceso": "Procesar dimensión Entidad Pública",                                 │
│   "tipo_carga_proceso": "Carga Inicial",                                                  │
│   "fecha_fin_extraccion": null,                                                           │
│   "observaciones": null,                                                                  │
│   "estado_registro": "Activo",                                                            │
│   "fecha_creacion": "2026-01-31 12:14:24",                                                │
│   "descripcion_proceso": "Proceso que gestiona la dimensión Entidad Pública de Colombia", │
│   "id_proceso": "317d33d8-025f-4eb1-baea-097873cc306d",                                   │
│   "id_proceso_padre": "e3153df7-92ff-448a-ac5b-61031adfa4f3",                             │
│   "fecha_inicio_extraccion": null,                                                        │
│   "email_responsable_proceso": "sirghoro@gmail.com",                                      │
│   "estado_ultima_ejecucion": "Correcto",                                                  │
│   "usuario_responsable": "postgres",                                                      │
│   "fecha_ultima_modificacion": null                                                       │
│ }                                                                                         │
╰───────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────── Registro de log ─────────────────────╮
│ {                                                         │
│   "id_ejecucion": "0bb38257-7248-4088-aec5-c44709b6fddc", │
│   "id_proceso": "317d33d8-025f-4eb1-baea-097873cc306d",   │
│   "fecha_inicio_ejecucion": "2026-02-02 18:38:34",        │
│   "fecha_fin_ejecucion": "2026-02-02 18:38:58",           │
│   "estado": "Correcto"                                    │
│ }                                                         │
╰───────────────────────────────────────────────────────────╯

(Denied) Denied by the resource provider.
Code: Denied
Message: Denied by the resource provider.

**Paso 5.2.** Simular el proceso ETL para la dimensión *Entidad Seccional*.

In [6]:
inicio_ejecucion = datetime.now()
mensaje = None
log_dim_entsec = None

proc_dim_entsec = piloto_framework()
proc_dim_entsec.configurar_proceso(id_proceso_dim_entidad_sec)
proc_dim_entsec.ver_datos_proceso()

if proc_dim_entsec.proceso._error_precedencias:
    log_dim_entsec = proc_dim_entsec.registro_log(
        inicio_ejecucion,
        EstadoEjecucion.ERROR,
        "Error de ejecución por precedencias: verifique el estado de la última ejecución de los procesos precedentes.",
        0,
    )
if proc_dim_entsec.proceso._ejecutar:
    try:
        par_tabla_fuente = proc_principal.parametros.get("tabla_destino_p1", "")
        par_tabla_destino = proc_principal.parametros.get(
            "tabla_destino_dim_entidad_sec", ""
        )
        consulta = f"select * from {par_tabla_fuente}"
        datos_fuente = pd.read_sql(consulta, con=proc_dim_entsec.bd_motor)
        entidades_seccionales = datos_fuente[
            ["id_de_la_entidad_seccional", "entidad_seccional"]
        ].drop_duplicates()
        # entidades_seccionales["sk_entidad_seccional"] = [uuid.uuid4() for _ in range(len(entidades_seccionales.index))]
        entidades_seccionales = entidades_seccionales.assign(
            sk_entidad_seccional=range(1, len(entidades_seccionales) + 1)
        )
        entidades_seccionales["fecha_creacion"] = datetime.now()
        entidades_seccionales["fecha_ultima_modificacion"] = None
        entidades_seccionales.rename(
            columns={
                "id_de_la_entidad_seccional": "id_entidad_seccional",
                "entidad_seccional": "nombre_entidad_seccional",
            },
            inplace=True,
        )
        columnas = [
            "sk_entidad_seccional",
            "id_entidad_seccional",
            "nombre_entidad_seccional",
            "fecha_creacion",
            "fecha_ultima_modificacion",
        ]
        entidades_seccionales = entidades_seccionales[columnas]
        grupo_1 = re.search(r"(?<=[.])\w+", par_tabla_destino)
        grupo_2 = re.search(r"(?<![.])\w+", par_tabla_destino)
        tabla: str | None = None
        esquema: str | None = None
        if grupo_1:
            tabla = grupo_1.group(0)
            esquema = grupo_2.group(0)
        elif grupo_2:
            tabla = grupo_2.group(0)
        else:
            tabla = ""
        tipo_carga = (
            "append"
            if proc_dim_entsec.proceso.tipo_carga_proceso == TipoCarga.INC
            else "replace"
        )
        entidades_seccionales.to_sql(
            name=tabla,
            schema=esquema,
            con=proc_dim_entsec.bd_motor,
            if_exists=tipo_carga,
            index=False,
        )
        estado_ejecucion = EstadoEjecucion.CORRECTO
        registros_procesados = len(entidades_seccionales)
    except Exception as error:
        estado_ejecucion = EstadoEjecucion.ERROR
        mensaje = f"Error en la ejecución: {error}"
        registros_procesados = 0
    finally:
        log_dim_entsec = proc_dim_entsec.registro_log(
            inicio_ejecucion, estado_ejecucion, mensaje, registros_procesados
        )

╭─────────────────────────────────── Detalles del proceso ────────────────────────────────────╮
│ {                                                                                           │
│   "nombre_proceso": "Procesar dimensión Entidad Seccional",                                 │
│   "tipo_carga_proceso": "Carga Inicial",                                                    │
│   "fecha_fin_extraccion": null,                                                             │
│   "observaciones": null,                                                                    │
│   "estado_registro": "Activo",                                                              │
│   "fecha_creacion": "2026-01-31 12:14:24",                                                  │
│   "descripcion_proceso": "Proceso que gestiona la dimensión Entidad Seccional de Colombia", │
│   "id_proceso": "57a54413-72e6-4a9e-bcb7-61044f447401",                                     │
│   "id_proceso_padre": "e3153df7-92ff-448a-ac5b-61031adfa4f3",                               │
│   "fecha_inicio_extraccion": null,                                                          │
│   "email_responsable_proceso": "sirghoro@gmail.com",                                        │
│   "estado_ultima_ejecucion": "Correcto",                                                    │
│   "usuario_responsable": "postgres",                                                        │
│   "fecha_ultima_modificacion": null                                                         │
│ }                                                                                           │
╰─────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────── Registro de log ─────────────────────╮
│ {                                                         │
│   "id_ejecucion": "71de951a-3443-42d5-be4b-d2b00dfd9ff2", │
│   "id_proceso": "57a54413-72e6-4a9e-bcb7-61044f447401",   │
│   "fecha_inicio_ejecucion": "2026-02-02 18:39:00",        │
│   "fecha_fin_ejecucion": "2026-02-02 18:39:29",           │
│   "estado": "Correcto"                                    │
│ }                                                         │
╰───────────────────────────────────────────────────────────╯

(Denied) Denied by the resource provider.
Code: Denied
Message: Denied by the resource provider.

**Paso 5.3.** Simular el proceso ETL para la dimensión *Departamento*.

In [7]:
inicio_ejecucion = datetime.now()
mensaje = None
log_dim_depto = None

proc_dim_depto = piloto_framework()
proc_dim_depto.configurar_proceso(id_proceso_dim_departamento)
proc_dim_depto.ver_datos_proceso()

if proc_dim_depto.proceso._error_precedencias:
    log_dim_depto = proc_dim_depto.registro_log(
        inicio_ejecucion,
        EstadoEjecucion.ERROR,
        "Error de ejecución por precedencias: verifique el estado de la última ejecución de los procesos precedentes.",
        0,
    )
if proc_dim_depto.proceso._ejecutar:
    try:
        par_tabla_fuente = proc_principal.parametros.get("tabla_destino_p1", "")
        par_tabla_destino = proc_principal.parametros.get(
            "tabla_destino_dim_departamento", ""
        )
        consulta = f"select * from {par_tabla_fuente}"
        datos_fuente = pd.read_sql(consulta, con=proc_dim_depto.bd_motor)
        departamentos = datos_fuente[
            ["divipola_departamento", "departamento"]
        ].drop_duplicates()
        # departamentos["sk_departamento"] = [uuid.uuid4() for _ in range(len(departamentos.index))]
        departamentos = departamentos.assign(
            sk_departamento=range(1, len(departamentos) + 1)
        )
        departamentos["fecha_creacion"] = datetime.now()
        departamentos["fecha_ultima_modificacion"] = None
        departamentos.rename(
            columns={
                "divipola_departamento": "id_departamento",
                "departamento": "nombre_departamento",
            },
            inplace=True,
        )
        columnas = [
            "sk_departamento",
            "id_departamento",
            "nombre_departamento",
            "fecha_creacion",
            "fecha_ultima_modificacion",
        ]
        departamentos = departamentos[columnas]
        grupo_1 = re.search(r"(?<=[.])\w+", par_tabla_destino)
        grupo_2 = re.search(r"(?<![.])\w+", par_tabla_destino)
        tabla: str | None = None
        esquema: str | None = None
        if grupo_1:
            tabla = grupo_1.group(0)
            esquema = grupo_2.group(0)
        elif grupo_2:
            tabla = grupo_2.group(0)
        else:
            tabla = ""
        tipo_carga = (
            "append"
            if proc_dim_depto.proceso.tipo_carga_proceso == TipoCarga.INC
            else "replace"
        )
        departamentos.to_sql(
            name=tabla,
            schema=esquema,
            con=proc_dim_depto.bd_motor,
            if_exists=tipo_carga,
            index=False,
        )
        estado_ejecucion = EstadoEjecucion.CORRECTO
        registros_procesados = len(departamentos)
    except Exception as error:
        estado_ejecucion = EstadoEjecucion.ERROR
        mensaje = f"Error en la ejecución: {error}"
        registros_procesados = 0
    finally:
        log_dim_depto = proc_dim_depto.registro_log(
            inicio_ejecucion, estado_ejecucion, mensaje, registros_procesados
        )

╭───────────────────────────────── Detalles del proceso ─────────────────────────────────╮
│ {                                                                                      │
│   "nombre_proceso": "Procesar dimensión Departamento",                                 │
│   "tipo_carga_proceso": "Carga Inicial",                                               │
│   "fecha_fin_extraccion": null,                                                        │
│   "observaciones": null,                                                               │
│   "estado_registro": "Activo",                                                         │
│   "fecha_creacion": "2026-01-31 12:14:24",                                             │
│   "descripcion_proceso": "Proceso que gestiona la dimensión Departamento de Colombia", │
│   "id_proceso": "83669434-13d7-40e3-92e3-fd67e24718e1",                                │
│   "id_proceso_padre": "e3153df7-92ff-448a-ac5b-61031adfa4f3",                          │
│   "fecha_inicio_extraccion": null,                                                     │
│   "email_responsable_proceso": "sirghoro@gmail.com",                                   │
│   "estado_ultima_ejecucion": "Correcto",                                               │
│   "usuario_responsable": "postgres",                                                   │
│   "fecha_ultima_modificacion": null                                                    │
│ }                                                                                      │
╰────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────── Registro de log ─────────────────────╮
│ {                                                         │
│   "id_ejecucion": "5ac8262e-0210-4025-9940-2bc04755df57", │
│   "id_proceso": "83669434-13d7-40e3-92e3-fd67e24718e1",   │
│   "fecha_inicio_ejecucion": "2026-02-02 18:39:30",        │
│   "fecha_fin_ejecucion": "2026-02-02 18:39:56",           │
│   "estado": "Correcto"                                    │
│ }                                                         │
╰───────────────────────────────────────────────────────────╯

(Denied) Denied by the resource provider.
Code: Denied
Message: Denied by the resource provider.

**Paso 5.4.** Simular el proceso ETL para la tabla de hechos *Proceso Judical Detallado*.

In [8]:
inicio_ejecucion = datetime.now()
mensaje = None
log_fact_procjud_det = None

proc_fact_procjud_det = piloto_framework()
proc_fact_procjud_det.configurar_proceso(id_proceso_fact_procjud_det)
proc_fact_procjud_det.ver_datos_proceso()

if proc_fact_procjud_det.proceso._error_precedencias:
    log_fact_procjud_det = proc_fact_procjud_det.registro_log(
        inicio_ejecucion,
        EstadoEjecucion.ERROR,
        "Error de ejecución por precedencias: verifique el estado de la última ejecución de los procesos precedentes.",
        0,
    )
if proc_fact_procjud_det.proceso._ejecutar:
    try:
        par_tabla_fuente = proc_principal.parametros.get("tabla_destino_p1", "")
        par_tabla_destino = proc_principal.parametros.get(
            "tabla_destino_hechos_procjud_det", ""
        )
        par_tabla_dim_depto = proc_principal.parametros.get(
            "tabla_destino_dim_departamento", ""
        )
        par_tabla_dim_entsec = proc_principal.parametros.get(
            "tabla_destino_dim_entidad_sec", ""
        )
        par_tabla_dim_entpub = proc_principal.parametros.get(
            "tabla_destino_dim_entidad_pub", ""
        )
        if len(par_tabla_destino) > 0:
            consulta = f"delete from {par_tabla_destino}"
            if (
                proc_fact_procjud_det.proceso.tipo_carga_proceso == TipoCarga.INC
                and ( not proc_fact_procjud_det.proceso.fecha_inicio_extraccion
                or not proc_fact_procjud_det.proceso.fecha_fin_extraccion )
            ):
                raise Exception("No se puede ejecutar la carga incremental porque la Fecha de inicio y/o fin no están definidas")
            else:
                consulta = consulta + f" where fecha_admision between '{proc_fact_procjud_det.proceso.fecha_inicio_extraccion}' and '{proc_fact_procjud_det.proceso.fecha_fin_extraccion}'"
            
            proc_fact_procjud_det.bd_sesion.exec(sqlalchemy.text(consulta))
            proc_fact_procjud_det.bd_sesion.commit()
        
        consulta = f"select * from {par_tabla_fuente}"
        if proc_fact_procjud_det.proceso.tipo_carga_proceso == TipoCarga.INC:
            consulta = consulta + f" where fecha_de_admisi_n_o between '{proc_fact_procjud_det.proceso.fecha_inicio_extraccion}' and '{proc_fact_procjud_det.proceso.fecha_fin_extraccion}'"
        
        datos_fuente = pd.read_sql(consulta, con=proc_fact_procjud_det.bd_motor)
        procesos = datos_fuente[
            [
                "c_digo_nico_del_proceso",
                "contraparte_persona_natural",
                "contraparte_persona_jur_dica",
                "contraparte_entidad",
                "fecha_de_admisi_n_o",
                "valor_econ_mico_indexado",
                "instancia_proceso_gesti_n",
                "estado_del_proceso_para_la",
                "actuaci_n_de_terminaci_n",
                "id_de_la_entidad",
                "id_de_la_entidad_seccional",
                "divipola_departamento",
            ]
        ]
        dim_departamento = pd.read_sql(
            f"select sk_departamento, id_departamento from {par_tabla_dim_depto}",
            con=proc_fact_procjud_det.bd_motor,
        )
        dim_entidad_pub = pd.read_sql(
            f"select sk_entidad_publica, id_entidad_publica from {par_tabla_dim_entpub}",
            con=proc_fact_procjud_det.bd_motor,
        )
        dim_entidad_sec = pd.read_sql(
            f"select sk_entidad_seccional, id_entidad_seccional from {par_tabla_dim_entsec}",
            con=proc_fact_procjud_det.bd_motor,
        )

        procesos = procesos.merge(
            dim_departamento,
            how="left",
            left_on="divipola_departamento",
            right_on="id_departamento",
        )
        procesos = procesos.merge(
            dim_entidad_pub,
            how="left",
            left_on="id_de_la_entidad",
            right_on="id_entidad_publica",
        )
        procesos = procesos.merge(
            dim_entidad_sec,
            how="left",
            left_on="id_de_la_entidad_seccional",
            right_on="id_entidad_seccional",
        )
        procesos["fecha_creacion"] = datetime.now()

        procesos.rename(
            columns={
                "c_digo_nico_del_proceso": "codigo_proceso",
                "contraparte_persona_jur_dica": "contraparte_persona_jur_dica",
                "fecha_de_admisi_n_o": "fecha_admision",
                "valor_econ_mico_indexado": "valor_economico_indexado",
                "instancia_proceso_gesti_n": "instancia_proceso_gesti_n",
                "estado_del_proceso_para_la": "estado_proceso_actuacion",
                "actuaci_n_de_terminaci_n": "actuacion_terminacion",
            },
            inplace=True,
        )
        columnas = [
            "sk_departamento",
            "sk_entidad_publica",
            "sk_entidad_seccional",
            "codigo_proceso",
            "contraparte_persona_natural",
            "contraparte_persona_jur_dica",
            "contraparte_entidad",
            "fecha_admision",
            "valor_economico_indexado",
            "instancia_proceso_gesti_n",
            "estado_proceso_actuacion",
            "actuacion_terminacion",
            "fecha_creacion",
        ]
        procesos = procesos[columnas]
        grupo_1 = re.search(r"(?<=[.])\w+", par_tabla_destino)
        grupo_2 = re.search(r"(?<![.])\w+", par_tabla_destino)
        tabla: str | None = None
        esquema: str | None = None
        if grupo_1:
            tabla = grupo_1.group(0)
            esquema = grupo_2.group(0)
        elif grupo_2:
            tabla = grupo_2.group(0)
        else:
            tabla = ""
        tipo_carga = (
            "append"
            if proc_fact_procjud_det.proceso.tipo_carga_proceso == TipoCarga.INC
            else "replace"
        )
        procesos.to_sql(
            name=tabla,
            schema=esquema,
            con=proc_fact_procjud_det.bd_motor,
            if_exists=tipo_carga,
            index=False,
        )
        estado_ejecucion = EstadoEjecucion.CORRECTO
        registros_procesados = len(procesos)
    except Exception as error:
        estado_ejecucion = EstadoEjecucion.ERROR
        mensaje = f"Error en la ejecución: {error}"
        registros_procesados = 0
    finally:
        log_fact_procjud_det = proc_fact_procjud_det.registro_log(
            inicio_ejecucion, estado_ejecucion, mensaje, registros_procesados
        )

╭───────────────────────────────────── Detalles del proceso ─────────────────────────────────────╮
│ {                                                                                              │
│   "nombre_proceso": "Procesar hechos Proceso Judicial Detallado",                              │
│   "tipo_carga_proceso": "Carga Incremental",                                                   │
│   "fecha_fin_extraccion": "2018-12-31",                                                        │
│   "observaciones": null,                                                                       │
│   "estado_registro": "Activo",                                                                 │
│   "fecha_creacion": "2026-01-31 12:14:24",                                                     │
│   "descripcion_proceso": "Proceso que gestiona la tabla de hechos Proceso Judicial Detallado", │
│   "id_proceso": "782045a8-59f1-4411-9a7c-145eeefb6d2d",                                        │
│   "id_proceso_padre": "e3153df7-92ff-448a-ac5b-61031adfa4f3",                                  │
│   "fecha_inicio_extraccion": "2018-09-01",                                                     │
│   "email_responsable_proceso": "sirghoro@gmail.com",                                           │
│   "estado_ultima_ejecucion": "Correcto",                                                       │
│   "usuario_responsable": "postgres",                                                           │
│   "fecha_ultima_modificacion": null                                                            │
│ }                                                                                              │
╰────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────── Registro de log ─────────────────────╮
│ {                                                         │
│   "id_ejecucion": "17aba4e9-d3a7-40f7-b821-aba9c97c557d", │
│   "id_proceso": "782045a8-59f1-4411-9a7c-145eeefb6d2d",   │
│   "fecha_inicio_ejecucion": "2026-02-02 18:39:57",        │
│   "fecha_fin_ejecucion": "2026-02-02 18:40:06",           │
│   "estado": "Correcto"                                    │
│ }                                                         │
╰───────────────────────────────────────────────────────────╯

(Denied) Denied by the resource provider.
Code: Denied
Message: Denied by the resource provider.

**Paso 5.5.** Simular el proceso ETL para la tabla de hechos *Proceso Judical Resumido*.

In [9]:
inicio_ejecucion = datetime.now()
mensaje = None
log_fact_procjud_res = None

proc_fact_procjud_res = piloto_framework()
proc_fact_procjud_res.configurar_proceso(id_proceso_fact_procjud_res)
proc_fact_procjud_res.ver_datos_proceso()

if proc_fact_procjud_res.proceso._error_precedencias:
    log_fact_procjud_res = proc_fact_procjud_res.registro_log(
        inicio_ejecucion,
        EstadoEjecucion.ERROR,
        "Error de ejecución por precedencias: verifique el estado de la última ejecución de los procesos precedentes.",
        0,
    )
if proc_fact_procjud_res.proceso._ejecutar:
    try:
        par_tabla_fuente = proc_principal.parametros.get("tabla_destino_hechos_procjud_det", "")
        par_tabla_destino = proc_principal.parametros.get(
            "tabla_destino_hechos_procjud_res", ""
        )
        if len(par_tabla_destino) > 0:
            consulta = f"delete from {par_tabla_destino}"
            if (
                proc_fact_procjud_res.proceso.tipo_carga_proceso == TipoCarga.INC
                and ( not proc_fact_procjud_res.proceso.fecha_inicio_extraccion
                or not proc_fact_procjud_res.proceso.fecha_fin_extraccion )
            ):
                raise Exception("No se puede ejecutar la carga incremental porque la Fecha de inicio y/o fin no están definidas")
            else:
                consulta = consulta + f" where fecha_admision between '{proc_fact_procjud_res.proceso.fecha_inicio_extraccion}' and '{proc_fact_procjud_res.proceso.fecha_fin_extraccion}'"
            
            proc_fact_procjud_res.bd_sesion.exec(sqlalchemy.text(consulta))
            proc_fact_procjud_res.bd_sesion.commit()
        
        consulta = f"select * from {par_tabla_fuente}"
        if proc_fact_procjud_res.proceso.tipo_carga_proceso == TipoCarga.INC:
            consulta = consulta + f" where fecha_admision between '{proc_fact_procjud_res.proceso.fecha_inicio_extraccion}' and '{proc_fact_procjud_res.proceso.fecha_fin_extraccion}'"
        
        datos_fuente = pd.read_sql(consulta, con=proc_fact_procjud_res.bd_motor)
        datos_fuente['valor_economico_indexado'] = datos_fuente['valor_economico_indexado'].astype(float)
        procesos = datos_fuente.groupby([
            "sk_departamento",
            "sk_entidad_publica",
            "sk_entidad_seccional",
            "fecha_admision",
            "estado_proceso_actuacion"
        ]).agg({'codigo_proceso':'count', 'valor_economico_indexado': 'sum'}).reset_index()
        procesos["fecha_creacion"] = datetime.now()
        procesos.rename(
            columns={
                "codigo_proceso": "cantidad_procesos"
            },
            inplace=True,
        )
        grupo_1 = re.search(r"(?<=[.])\w+", par_tabla_destino)
        grupo_2 = re.search(r"(?<![.])\w+", par_tabla_destino)
        tabla: str | None = None
        esquema: str | None = None
        if grupo_1:
            tabla = grupo_1.group(0)
            esquema = grupo_2.group(0)
        elif grupo_2:
            tabla = grupo_2.group(0)
        else:
            tabla = ""
        tipo_carga = (
            "append"
            if proc_fact_procjud_res.proceso.tipo_carga_proceso == TipoCarga.INC
            else "replace"
        )
        procesos.to_sql(
            name=tabla,
            schema=esquema,
            con=proc_fact_procjud_res.bd_motor,
            if_exists=tipo_carga,
            index=False,
        )
        estado_ejecucion = EstadoEjecucion.CORRECTO
        registros_procesados = len(procesos)
    except Exception as error:
        estado_ejecucion = EstadoEjecucion.ERROR
        mensaje = f"Error en la ejecución: {error}"
        registros_procesados = 0
    finally:
        log_fact_procjud_res = proc_fact_procjud_res.registro_log(
            inicio_ejecucion, estado_ejecucion, mensaje, registros_procesados
        )

╭──────────────────────────────────── Detalles del proceso ─────────────────────────────────────╮
│ {                                                                                             │
│   "nombre_proceso": "Procesar hechos Proceso Judicial Resumido",                              │
│   "tipo_carga_proceso": "Carga Incremental",                                                  │
│   "fecha_fin_extraccion": "2018-12-31",                                                       │
│   "observaciones": null,                                                                      │
│   "estado_registro": "Activo",                                                                │
│   "fecha_creacion": "2026-01-31 12:14:24",                                                    │
│   "descripcion_proceso": "Proceso que gestiona la tabla de hechos Proceso Judicial Resumido", │
│   "id_proceso": "e72b69f0-d128-440a-ad97-03616d9b85c6",                                       │
│   "id_proceso_padre": "e3153df7-92ff-448a-ac5b-61031adfa4f3",                                 │
│   "fecha_inicio_extraccion": "2018-09-01",                                                    │
│   "email_responsable_proceso": "sirghoro@gmail.com",                                          │
│   "estado_ultima_ejecucion": "Correcto",                                                      │
│   "usuario_responsable": "postgres",                                                          │
│   "fecha_ultima_modificacion": null                                                           │
│ }                                                                                             │
╰───────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────── Registro de log ─────────────────────╮
│ {                                                         │
│   "id_ejecucion": "3893ee13-8601-408f-ae97-45627e5e1804", │
│   "id_proceso": "e72b69f0-d128-440a-ad97-03616d9b85c6",   │
│   "fecha_inicio_ejecucion": "2026-02-02 18:40:08",        │
│   "fecha_fin_ejecucion": "2026-02-02 18:40:10",           │
│   "estado": "Correcto"                                    │
│ }                                                         │
╰───────────────────────────────────────────────────────────╯

(Denied) Denied by the resource provider.
Code: Denied
Message: Denied by the resource provider.

**Paso 5.6.** Cierre de la simulación.
En este paso se registra el log de ejecución para el proceso agrupador.

In [10]:
estados = [
    log_dim_entpub.estado_ejecucion if log_dim_entpub else EstadoEjecucion.CORRECTO,
    log_dim_entsec.estado_ejecucion if log_dim_entsec else EstadoEjecucion.CORRECTO,
    log_dim_depto.estado_ejecucion if log_dim_depto else EstadoEjecucion.CORRECTO,
    log_fact_procjud_det.estado_ejecucion if log_fact_procjud_det else EstadoEjecucion.CORRECTO,
    log_fact_procjud_res.estado_ejecucion if log_fact_procjud_res else EstadoEjecucion.CORRECTO,
]

if estados.count(EstadoEjecucion.ERROR) > 0:
    estado_ejecucion = EstadoEjecucion.ERROR
else:
    estado_ejecucion = EstadoEjecucion.CORRECTO

registros_procesados = (
    (log_dim_entpub.registros_procesados if log_dim_entpub else 0)
    + (log_dim_entsec.registros_procesados if log_dim_entsec else 0)
    + (log_dim_depto.registros_procesados if log_dim_depto else 0)
    + (log_fact_procjud_det.registros_procesados if log_fact_procjud_det else 0)
    + (log_fact_procjud_res.registros_procesados if log_fact_procjud_res else 0)
)
mensajes = [
    log_dim_entpub.nombre_proceso + ": " + log_dim_entpub.mensaje_ejecucion
    if log_dim_entpub and log_dim_entpub.mensaje_ejecucion
    else None,
    log_dim_entsec.nombre_proceso + ": " + log_dim_entsec.mensaje_ejecucion
    if log_dim_entsec and log_dim_entsec.mensaje_ejecucion
    else None,
    log_dim_depto.nombre_proceso + ": " + log_dim_depto.mensaje_ejecucion
    if log_dim_depto and log_dim_depto.mensaje_ejecucion
    else None,
    log_fact_procjud_det.nombre_proceso + ": " + log_fact_procjud_det.mensaje_ejecucion
    if log_fact_procjud_det and log_fact_procjud_det.mensaje_ejecucion
    else None,
    log_fact_procjud_res.nombre_proceso + ": " + log_fact_procjud_res.mensaje_ejecucion
    if log_fact_procjud_res and log_fact_procjud_res.mensaje_ejecucion
    else None,
]
mensajes = [_msj for _msj in mensajes if _msj is not None]
mensaje = "\r\n".join(mensajes)
log_proc = proc_principal.registro_log(
    inicio_ejecucion_principal,
    estado_ejecucion,
    mensaje if len(mensaje) > 0 else None,
    registros_procesados,
)

╭───────────────────── Registro de log ─────────────────────╮
│ {                                                         │
│   "id_ejecucion": "997e034c-e9f2-4a5d-9ec2-a7ccf3a36972", │
│   "id_proceso": "e3153df7-92ff-448a-ac5b-61031adfa4f3",   │
│   "fecha_inicio_ejecucion": "2026-02-02 18:38:34",        │
│   "fecha_fin_ejecucion": "2026-02-02 18:40:11",           │
│   "estado": "Correcto"                                    │
│ }                                                         │
╰───────────────────────────────────────────────────────────╯

(Denied) Denied by the resource provider.
Code: Denied
Message: Denied by the resource provider.

**Paso 6.** Limpieza

In [11]:
import gc as _gc

for name in dir():
    if not name.startswith("_"):
        del globals()[name]
del name
_gc.collect()


1973